# 交通标志检测 - YOLOv8

本 notebook 用于在百度 AI Studio 上运行交通标志检测任务。

## 1. 安装依赖

In [ ]:
# 安装 ultralytics
!pip install ultralytics -q

## 2. 导入库

In [ ]:
import os
import csv
from pathlib import Path
from ultralytics import YOLO
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd

## 3. 查看数据目录结构

In [ ]:
# 查看数据目录
!ls -la data/

## 4. 创建数据配置文件

In [ ]:
data_yaml_content = """
path: data
train: train/images
val: val/images
test: test/images
nc: 15
names:
  0: Green Light
  1: Red Light
  2: Speed Limit 10
  3: Speed Limit 100
  4: Speed Limit 110
  5: Speed Limit 120
  6: Speed Limit 20
  7: Speed Limit 30
  8: Speed Limit 40
  9: Speed Limit 50
  10: Speed Limit 60
  11: Speed Limit 70
  12: Speed Limit 80
  13: Speed Limit 90
  14: Stop
"""

with open('data.yaml', 'w') as f:
    f.write(data_yaml_content)

print("data.yaml 创建完成")

## 5. 训练模型

In [ ]:
# 加载预训练模型
model = YOLO('yolov8n.pt')

# 训练参数
results = model.train(
    data='data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    name='traffic_signs',
    project='runs/detect',
    patience=10,
    save=True,
    device=0,  # 使用 GPU
    workers=4,
    verbose=True
)

## 6. 查看训练结果

In [ ]:
# 查看训练结果曲线
results_path = 'runs/detect/traffic_signs/results.png'
if os.path.exists(results_path):
    img = Image.open(results_path)
    plt.figure(figsize=(15, 10))
    plt.imshow(img)
    plt.axis('off')
    plt.show()

## 7. 验证模型

In [ ]:
# 加载最佳模型
best_model = YOLO('runs/detect/traffic_signs/weights/best.pt')

# 在验证集上评估
metrics = best_model.val(data='data.yaml')

print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")

## 8. 对测试集进行预测并生成提交文件

In [ ]:
# 加载最佳模型
model = YOLO('runs/detect/traffic_signs/weights/best.pt')

# 测试集路径
test_dir = Path('data/test/images')
image_paths = sorted([p for p in test_dir.iterdir() if p.is_file()])

print(f"测试集图片数量: {len(image_paths)}")

In [ ]:
# 进行预测并生成 submission.csv
output_file = 'submission.csv'

with open(output_file, 'w', encoding='utf-8', newline='') as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=['image_id', 'class_id', 'x_center', 'y_center', 'width', 'height', 'confidence'],
    )
    writer.writeheader()
    
    # 批量预测
    results = model.predict(
        source=[str(p) for p in image_paths],
        conf=0.001,
        save=False,
        verbose=True
    )
    
    for result in results:
        image_id = Path(result.path).name
        if result.boxes is None:
            continue
        for box in result.boxes:
            x_center, y_center, width, height = box.xywhn[0].tolist()
            writer.writerow(
                {
                    'image_id': image_id,
                    'class_id': int(box.cls[0].item()),
                    'x_center': x_center,
                    'y_center': y_center,
                    'width': width,
                    'height': height,
                    'confidence': float(box.conf[0].item()),
                }
            )

print(f"\n提交文件已生成: {output_file}")

## 9. 查看提交文件

In [ ]:
# 读取并显示提交文件的前几行
submission_df = pd.read_csv('submission.csv')
print(f"提交文件总行数: {len(submission_df)}")
print("\n前10行:")
print(submission_df.head(10))

## 10. 可视化部分预测结果

In [ ]:
# 随机选择一些测试图片进行可视化
import random

sample_images = random.sample([str(p) for p in image_paths], min(4, len(image_paths)))

results = model.predict(source=sample_images, conf=0.25, save=False)

fig, axes = plt.subplots(2, 2, figsize=(15, 15))
axes = axes.flatten()

for idx, result in enumerate(results):
    img = Image.open(result.path)
    axes[idx].imshow(img)
    axes[idx].set_title(Path(result.path).name)
    axes[idx].axis('off')
    
    # 绘制预测框
    if result.boxes is not None:
        for box in result.boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            cls_id = int(box.cls[0].item())
            conf = float(box.conf[0].item())
            
            width_img, height_img = img.size
            rect = plt.Rectangle(
                (x1, y1), x2-x1, y2-y1,
                fill=False, edgecolor='red', linewidth=2
            )
            axes[idx].add_patch(rect)
            axes[idx].text(
                x1, y1-5,
                f'{model.names[cls_id]}: {conf:.2f}',
                color='red', fontsize=10,
                bbox=dict(facecolor='white', alpha=0.7)
            )

plt.tight_layout()
plt.show()

## 11. 下载提交文件

运行以下代码下载 submission.csv 文件：

In [ ]:
from IPython.display import FileLink
FileLink('submission.csv')